# 1. Setup

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.extractor import Extractor
from scripts.utils import build_hdf5, generate_plys, read_metadata

/home/verve/Storage/Datasets/MorphCell/workspaces/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2. Run

In [2]:
MESH_DIR = PROJECT_ROOT / "data" / "meshes"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

extractor = Extractor(mesh_dir=MESH_DIR)

In [ ]:
# single features
single_df = extractor.extract_all_single_features()
print(single_df.shape)
single_df.head()

In [ ]:
# pair features
pair_df = extractor.extract_pair_features(single_df)
print(pair_df.shape)
pair_df.head()

In [ ]:
# save outputs
single_out = OUTPUT_DIR / "single_cell_features.csv"
pair_out = OUTPUT_DIR / "paired_cell_features.csv"

single_df.to_csv(single_out, index=False)
pair_df.to_csv(pair_out, index=False)

# 3. Generate point clouds and HDF5

In [3]:
METADATA_PATH = PROJECT_ROOT / "data" / "metadata.csv"
POINTCLOUD_DIR = PROJECT_ROOT / "data" / "pointclouds"
NUM_POINTS = 2048
SEED = 42
MESH_SCALE = 1.0  # retain the 1/20-scale mesh coordinates

rows = read_metadata(METADATA_PATH)
generate_plys(
    MESH_DIR,
    POINTCLOUD_DIR,
    rows,
    num_points=NUM_POINTS,
    seed=SEED,
    mesh_scale=MESH_SCALE,
)

In [4]:
build_hdf5(
    POINTCLOUD_DIR,
    OUTPUT_DIR,
    rows,
    num_points=NUM_POINTS,
    mesh_scale=MESH_SCALE,
)